### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

A decision model doesn't write text. It reads an input, looks at the options you give it, and picks one with a probability. `FastDecisionModel` turns an LLM into one: it adds a small head that reads every question about an input in one pass and scores each option.

Change `model_name` to `unsloth/Llama-3.2-3B-Instruct` or `unsloth/gemma-4-E4B-it` to try other models.

In [2]:
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
import torch

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048, # Longest input. Long inputs keep the question and options.
    load_in_4bit = True, # 4 bit quantization to reduce memory
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/models/rl.py:1303: UserWarning: Torchaudio's I/O functions now support per-call backend dispatch. Importing backend implementation directly is no longer guaranteed to work. Please use `backend` keyword with load/save/info function, instead of calling the underlying implementation directly.
  if getattr(module, "prepare_peft_model", None) is original:


==((====))==  Unsloth 2026.10.3: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update a small amount of parameters! The new decision head always trains.

In [3]:
model = FastDecisionModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We use the [typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) dataset. Each row has a `state` (the input), the `questions` to decide about it, and the `gold` answers. There are three kinds of questions:

* `choice`: pick one option, like which team should handle a ticket.
* `noul`: yes or no.
* `score`: pick a level, like how urgent something is.

Let's see how row 0 looks like!

In [4]:
from datasets import load_dataset
import json

dataset = load_dataset("LocalLLaMA/typed-decisions", "all", split = "train")
print(dataset[0]["state"][:500])
print(json.dumps(json.loads(dataset[0]["questions"]), indent = 2)[:1000])
print(dataset[0]["gold"])

README.md:   0%|          | 0.00/15.5k [00:00<?, ?B/s]

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

all/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

{"agent": {"autonomy": "checkpointed", "model": "internal-agent-v4"}, "constraints": ["Do not touch customer data outside the named accounts"], "task": "Delete personal data for the accounts in the erasure queue.", "trace_summary": {"constraint_violations": 0, "duration_s": 128.5, "irreversible_actions": 0, "steps": 7, "tool_errors": 1}}
{
  "action": {
    "criteria": {
      "continue": "Let the agent proceed without interruption.",
      "human_review": "Queue this trace for a human to review.",
      "observe": "Keep running, but flag the trace for later sampling.",
      "stop": "Halt the agent now."
    },
    "instructions": "What should the observability system do with this trace?",
    "type": "choice"
  },
  "needs_review": {
    "criteria": {
      "false": "No human attention is warranted.",
      "true": "A human should inspect this run."
    },
    "instructions": "This trace requires human review.",
    "type": "noul"
  },
  "outcome": {
    "criteria": {
      "failure"

`build_dataset` turns every row into one example holding all of its questions, and tells you how many decisions it skipped and why. To use your own data, give it a list of rows with the same `state`, `questions` and `gold` fields.

`split_holdout` keeps some rows out of training (80 rows, 400 decisions here), so we can check accuracy and calibrate the model later.

In [5]:
items, report = FastDecisionModel.build_dataset(dataset, tokenizer, model)
print(f"Skipped {report['skipped']} of {report['total']} decisions")

train_items, eval_items = FastDecisionModel.split_holdout(items, seed = 3407)
print(f"{len(train_items)} training rows, {len(eval_items)} held out")

Skipped 0 of 6000 decisions
1120 training rows, 80 held out


Let's check accuracy before training. The head is new, so it's about the same as guessing (about 30% accuracy in our runs).

In [6]:
FastDecisionModel.evaluate(model, tokenizer, eval_items)

{'accuracy': 0.3025,
 'ece': 0.06227386500686405,
 'loss': 1.2141682821512223,
 'record_accuracy': 0.0}

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps (about 1.7 epochs) to speed things up, which reached 75% to 78% test accuracy in our runs and took about 2 hours on a free T4 GPU (12 minutes on an RTX PRO 6000). The T4 has no bfloat16 and Qwen3.5 gives NaNs in pure float16, so Unsloth trains it in float32 there, which is why the T4 is slow. For the full run, set `num_train_epochs = 2` and remove `max_steps`: that is 70 steps here, and it reached 78% test accuracy in 25 minutes on an A100.

In [7]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = eval_items,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 10,
        # num_train_epochs = 2, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
        seed = 3407,
    ),
)

Unsloth: Qwen3.5 overflows under float16 autocast, so Clef trains without autocast, keeping float32 activations.


In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.562 GB.
5.154 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,120 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 64,511,492 of 3,072,856,068 (2.10% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.862913,0.861985


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-60/.unsloth-save-d9bjxrz6/tokenizer_config.json.


In [10]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

6224.6337 seconds used for training.
103.74 minutes used for training.
Peak reserved memory = 7.643 GB.
Peak reserved memory for training = 2.489 GB.
Peak reserved memory % of max memory = 52.486 %.
Peak reserved memory for training % of max memory = 17.092 %.


<a name="Inference"></a>
### Inference
First we calibrate the model on the held-out rows. Calibration adjusts the probabilities, so an answer given with 90% confidence is right about 90% of the time. `ece` is the calibration error, lower is better. After 60 steps we got 80% to 84% held-out accuracy and an `ece` of 0.02 to 0.06.

In [11]:
FastDecisionModel.calibrate(model, tokenizer, eval_items)

{'accuracy': 0.79,
 'ece': 0.04605473153293132,
 'loss': 1.150050092182355,
 'record_accuracy': 0.3375,
 'fitted_types': [0, 1, 2]}

Let's check accuracy on the dataset's test split, which the model never saw. We got 75% to 78% after 60 steps:

In [12]:
test = load_dataset("LocalLLaMA/typed-decisions", "all", split = "test")
test_items, _ = FastDecisionModel.build_dataset(test, tokenizer, model)
FastDecisionModel.evaluate(model, tokenizer, test_items)

{'accuracy': 0.754,
 'ece': 0.03662284298241141,
 'loss': 1.1740286591524491,
 'record_accuracy': 0.2825}

Now let's make some decisions! Give `predict` an input and your questions. `answer` is the option for `choice`, `True` or `False` for `noul`, and the level number for `score`.

In [13]:
FastDecisionModel.for_inference(model)
answers = FastDecisionModel.predict(
    model,
    tokenizer,
    "Hi, I was charged twice for invoice #4411. Please refund the duplicate today.",
    {
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "invoices, payments, refunds",
                "technical": "bugs, outages, errors",
                "sales": "pricing, new plans",
            },
        },
        "refund": {"type": "noul", "instructions": "Does the customer ask for a refund?"},
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this?",
            "criteria": ["not urgent", "soon", "today"],
        },
    },
)
for name, result in answers.items():
    print(name, result["answer"], {k: round(v, 3) for k, v in result["probabilities"].items()})

team billing {'billing': 0.997, 'technical': 0.003, 'sales': 0.0}
refund True {'true': 0.998, 'false': 0.002}
urgency 2 {'0': 0.0, '1': 0.044, '2': 0.956}


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model, use `save_pretrained` for a local save or `push_to_hub` for an online save. This saves the LoRA adapters, the decision head and its calibration.

In [ ]:
model.save_pretrained("qwen3.5-4b-decision")  # Local saving
model.push_to_hub("p2kalita/qwen3.5-4b-decision", token = "hf_xxxxxxxxxxxxx") # Online saving

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3.5-4b-decision/.unsloth-save-h_mfn1nv/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp8h931g3e/.unsloth-save-4yfdz_u0/tokenizer_config.json.


Unsloth: Saved the decision model to https://huggingface.co/p2kalita/qwen3.5-4b-decision


Now if you want to load the model we just saved, set `False` to `True`:

In [15]:
if True:
    from unsloth import FastDecisionModel
    model, tokenizer = FastDecisionModel.from_pretrained(
        model_name = "qwen3.5-4b-decision", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

==((====))==  Unsloth 2026.10.3: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

### Saving to float16

We also support saving to `float16` directly with `save_pretrained_merged`. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("p2kalita/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")